# Независимый локальный инференс

Этот notebook запускается в **новом ядре** после training notebooks. Обучающие ячейки исполнять не требуется. Веса, архитектура, POD-базисы, mean/std, геометрия и порядок FEM-узлов загружаются из одного `model.pt`. FEM solver для предсказания не вызывается.

Для демонстрации берём входные A,n,theta,k из сохранённого test. Истинное u используется только для оценки; `predict()` принимает только входные коэффициенты и параметры падающей волны. Новые A,n должны быть в том же порядке FEM-узлов.

**Статичная задача:** геометрия и исходная среда №0 фиксированы; A,n и DG0-материалы одинаковы во всех примерах. Меняются k и θ. Один medium_id=0, разбиение по frequency_id: 670/170/160 частот (5360/1360/1280 полей). 1000 частот × 8 направлений = 8000 полей на полосу. Этот эксперимент проверяет интерполяцию параметров волны на одной среде.

**Профиль:** выберите `POD_METHOD="full"` или `"randomized"` в первой ячейке. Данные общие; пути результатов разделены. Сохранённые outputs относятся к randomized. Это выбор алгоритма POD, а не отдельные версии Chebyshev/FNO.

**Результат:** Re(u_scattering), Im(u_scattering), Abs(u_scattering) в исходных FEM-узлах. Модуль вычисляется из восстановленного комплексного поля: `abs = sqrt(real**2 + imag**2)`; отдельный независимый выход для него не обучается. На рисунке три строки — Re, Im, Abs, три столбца — FEM, предсказание и абсолютная ошибка соответствующей величины. Для FEM и модели в каждой строке используется одна шкала цвета.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from neural_operator import settings as full_settings
from neural_operator.randomized import settings as randomized_settings
POD_METHOD = "randomized"  # "full" uses separate runs.
if POD_METHOD not in {"full", "randomized"}:
    raise ValueError("POD_METHOD must be full or randomized")
settings = randomized_settings if POD_METHOD == "randomized" else full_settings
BASE, N_FREQUENCIES, N_DIRECTIONS, DATASET_TAG = (settings.BASE, settings.N_FREQUENCIES,
    settings.N_DIRECTIONS, settings.DATASET_TAG)
LOW_DATA_DIR = settings.LOW_DATA_DIR
LOW_RUNS = settings.LOW_RUNS
SVD_RUNS, COMPARISON_RUNS = settings.SVD_RUNS, settings.COMPARISON_RUNS
DATA_DIR = LOW_DATA_DIR
RUNS = ROOT / 'neural_operator' / 'runs' / 'revision_2026-11-08' / 'static_low'
MODEL_RUNS = {
    'svd': RUNS / 'svd', 'chebyshev': RUNS / 'chebyshev',
    'fno': ROOT / 'neural_operator' / 'randomized' / 'runs' / 'low_frequency_static_1000x8' / 'fno',
}
import json
import numpy as np
import matplotlib.pyplot as plt
from neural_operator.datasets import load_dataset, write_dataset_info
from neural_operator.plotting import display_table, plot_prediction, plot_history


In [ ]:
from neural_operator.training import load_predictor
METHOD = 'svd'  # svd, chebyshev, fno
CHECKPOINT = MODEL_RUNS[METHOD] / 'model.pt'
predictor = load_predictor(CHECKPOINT, device='cpu')
dataset = load_dataset(DATA_DIR, mmap=True)
indices = dataset.split_indices()[2][:4]
index = int(indices[0])
components = predictor.predict_components(
    dataset.A[index], dataset.n[index], dataset.theta[index], dataset.k0[index])
real, imag, amplitude = components['real'], components['imag'], components['abs']
prediction = real + 1j * imag
print('Re(u_scattering):', real.shape, real.dtype)
print('Im(u_scattering):', imag.shape, imag.dtype)
print('Abs(u_scattering):', amplitude.shape, amplitude.dtype)
np.testing.assert_allclose(amplitude, np.abs(prediction))
print('Consistent FEM L2:', float(dataset.relative_l2(dataset.u_scat[index], prediction)))


In [ ]:
from time import perf_counter
from neural_operator.diagnostics import write_csv
from neural_operator.training import summarize
rows = []
METHODS = [METHOD]  # Add other methods after their training notebooks finish.
for method in METHODS:
    model = load_predictor(MODEL_RUNS[method] / 'model.pt')
    model.predict_dataset(dataset, indices[:1])  # warm-up
    started = perf_counter()
    batch = model.predict_dataset(dataset, indices)
    seconds = perf_counter() - started
    single = model.predict(dataset.A[index], dataset.n[index], dataset.theta[index], dataset.k0[index])
    difference = float(np.max(np.abs(single - batch[0])))
    np.testing.assert_allclose(single, batch[0], rtol=2e-5, atol=2e-6)
    rows.append({'method': method, **summarize(dataset.relative_l2(dataset.u_scat[indices], batch)),
                 'single_batch_max_difference': difference, 'batch_seconds': seconds})
display_table(rows)
OUT = BASE / 'runs' / f'inference_{DATASET_TAG}'
OUT.mkdir(parents=True, exist_ok=True)
write_csv(OUT / 'parity.csv', rows)
np.savez_compressed(OUT / 'prediction.npz', index=index, u_scat=prediction,
                    real=real, imag=imag, abs=amplitude,
                    dof_coords=predictor.coordinates, triangles=predictor.triangles)


In [ ]:
# Три строки: Re, Im, Abs; столбцы: FEM, модель, ошибка.
figure = plot_prediction(dataset, index, prediction, title=f'{METHOD}: independent inference')
figure.savefig(OUT / 'prediction.png', dpi=160)
plt.show()


Для внешнего входа загрузите npz с A,n,theta,k0 и передайте их в `predictor.predict(...)`; хранить обучающий датасет для этого не требуется. `predict_dataset()` дополнительно сверяет connectivity и координаты с checkpoint. Изменение геометрии и числа FEM-узлов требует отдельного протокола переноса и в текущий эксперимент не входит.

Для выдачи трёх вещественных массивов используйте `predictor.predict_components(...)`: ключи `real`, `imag`, `abs`. Метод `predict()` по-прежнему возвращает комплексное поле, из которого эти величины можно получить через `.real`, `.imag` и `np.abs(...)`. Файл `prediction.npz` содержит и `u_scat`, и все три массива. Объяснение вычислений: [методичка SVD/SNO](../docs/svd_sno_guide.md).
